# FAQ Chatbot Analysis and Implementation
This notebook analyzes the `customer_support_faq.csv` dataset and implements a simple TF-IDF based Chatbot to answer user questions.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. Load the Data
df = pd.read_csv('customer_support_faq.csv')
print(f"Dataset contains {df.shape[0]} questions across {df['category'].nunique()} categories.")
df.head()

## 1. Exploratory Data Analysis
Let's see the distribution of questions per category.

In [ ]:
df['category'].value_counts().plot(kind='bar', color='skyblue', edgecolor='black')
plt.title('Number of Questions per Category')
plt.xlabel('Category')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 2. Chatbot Implementation
For the chatbot, we'll use a basic Natural Language Processing (NLP) technique: **TF-IDF (Term Frequency-Inverse Document Frequency)** combined with **Cosine Similarity**.

This approach will:
1. Convert all FAQ questions into vector representations.
2. Convert the user's query into a vector.
3. Compare the user's query vector against all FAQ question vectors using cosine similarity to find the closest match.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Initialize the TF-IDF Vectorizer (removes standard English stop words)
vectorizer = TfidfVectorizer(stop_words='english')

# Fit and transform the FAQ questions into a TF-IDF matrix
tfidf_matrix = vectorizer.fit_transform(df['question'])

def get_chatbot_response(user_query, threshold=0.2):
    """
    Takes a user query and returns the most relevant FAQ answer.
    """
    # Vectorize the user query
    query_vec = vectorizer.transform([user_query])
    
    # Calculate cosine similarity between the query and all FAQ questions
    similarities = cosine_similarity(query_vec, tfidf_matrix).flatten()
    
    # Get the index of the most similar question
    best_match_idx = similarities.argmax()
    
    # Check if the highest similarity is above the threshold
    if similarities[best_match_idx] >= threshold:
        best_question = df.iloc[best_match_idx]['question']
        answer = df.iloc[best_match_idx]['answer']
        return f"**Matched FAQ:** {best_question}\n\n**Answer:** {answer}"
    else:
        return "I'm sorry, I couldn't find an exact answer to your question. Please try rephrasing or contact a live agent."

## 3. Testing the Chatbot
Let's test the chatbot with a few sample queries.

In [ ]:
test_queries = [
    "How can I track where my order is?",
    "I want to return an item I got as a gift without a receipt",
    "What happens if I forgot my password?",
    "Can you tell me a joke?"  # Expected to fail the threshold
]

for q in test_queries:
    print(f"\033[94mUser Query:\033[0m {q}")
    print(f"\033[92mChatbot:\033[0m {get_chatbot_response(q)}")
    print("-"*60)

## 4. Model Training and Export
To use this model in our backend engine, we will export the trained vectorizer and dataset using joblib.

In [ ]:
import joblib
import os

# Save the vectorizer, the tfidf_matrix, and the dataframe
model_data = {
    'vectorizer': vectorizer,
    'tfidf_matrix': tfidf_matrix,
    'df': df
}

joblib.dump(model_data, 'chatbot_model.joblib')
print('Model successfully trained and exported to chatbot_model.joblib')